# Phase 4 — Matched-recall evaluation, feature availability and review-capacity simulation

**Scope:** retrospective provider-level benchmark only. No prospective, claim-level, clinical, or CMS deployment validation. Thresholds fixed using validation data; test labels are used only for evaluation. All outputs are exploratory.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_curve, roc_auc_score, average_precision_score, confusion_matrix
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
OUT=ROOT/'outputs'; OUT.mkdir(exist_ok=True)
df=pd.read_csv(OUT/'provider_level_features.csv')
features=[c for c in df if c not in ('Provider','PotentialFraud','Label')]
X=df[features]; y=df.Label
idx_dev,idx_test=train_test_split(np.arange(len(df)),test_size=.20,random_state=42,stratify=y)
idx_train,idx_val=train_test_split(idx_dev,test_size=.25,random_state=43,stratify=y.iloc[idx_dev])
assert len(set(idx_train)&set(idx_val))==len(set(idx_train)&set(idx_test))==len(set(idx_val)&set(idx_test))==0
models={'Logistic regression':make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),LogisticRegression(class_weight='balanced',max_iter=1500,random_state=42)), 'Gradient boosting':make_pipeline(SimpleImputer(strategy='median'),GradientBoostingClassifier(random_state=42))}
for m in models.values(): m.fit(X.iloc[idx_train],y.iloc[idx_train])
print('Providers',len(df),'train',len(idx_train),'validation',len(idx_val),'test',len(idx_test),'prevalence',round(y.mean(),4))

Providers 5410 train 3246 validation 1082 test 1082 prevalence 0.0935


## 1. Prospective feature-availability audit
These flags are **analyst assessments**, not confirmations of availability from a payer data warehouse. All full-history provider aggregates are unsafe for a claimed claim-time prepayment deployment until re-engineered using strictly prior records and timestamped snapshots.

In [2]:
feature_audit=[]
for f in features:
    beneficiary=f in ['AvgBeneficiaryAge','AvgChronicConditions','PctDeceasedBeneficiaries','PctRenalDisease']
    feature_audit.append({'Feature':f,'HistoricalBenchmarkUse':'Yes','PrepaymentAvailabilityVerified':'No','PotentialLeakageMechanism':('Beneficiary status/comorbidities may reflect updates after index claim' if beneficiary else 'Aggregate may include claims or payments after index decision'), 'RequiredRemediation':('As-of join to eligibility/clinical information available before decision' if beneficiary else 'Calculate rolling provider history from records strictly earlier than claim submission'), 'ClaimTimeSafeAsCurrentlyConstructed':'No'})
audit=pd.DataFrame(feature_audit)
audit.to_csv(OUT/'phase4_feature_availability_audit.csv',index=False)
print(audit[['Feature','PrepaymentAvailabilityVerified','ClaimTimeSafeAsCurrentlyConstructed']].to_string(index=False))

                     Feature PrepaymentAvailabilityVerified ClaimTimeSafeAsCurrentlyConstructed
                   NumClaims                             No                                  No
      NumUniqueBeneficiaries                             No                                  No
             TotalReimbursed                             No                                  No
       AvgReimbursedPerClaim                             No                                  No
       StdReimbursedPerClaim                             No                                  No
          MaxReimbursedClaim                             No                                  No
             TotalDeductible                             No                                  No
        AvgClaimDurationDays                             No                                  No
         AvgLengthOfStayDays                             No                                  No
             AvgNumDiagCodes            

## 2. Validation-selected matched-target recall
Choose each model threshold on validation at a common target recall of 70%. Because of finite sample effects, observed **test** recall will differ. Report actual recall and FPR together; do not claim exact test matched recall.

In [3]:
def threshold_at_validation_recall(truth,scores,target=.70):
    fpr,tpr,thresholds=roc_curve(truth,scores)
    valid=np.where((tpr>=target)&np.isfinite(thresholds))[0]
    return float(thresholds[valid[np.argmin(fpr[valid])]])
truth=y.iloc[idx_test].to_numpy(); result=[]; pred_records=[]; test_scores={}
for name,model in models.items():
    vs=model.predict_proba(X.iloc[idx_val])[:,1]; ts=model.predict_proba(X.iloc[idx_test])[:,1]
    t=threshold_at_validation_recall(y.iloc[idx_val].to_numpy(),vs)
    vp=(vs>=t).astype(int); pred=(ts>=t).astype(int)
    vtn,vfp,vfn,vtp=confusion_matrix(y.iloc[idx_val],vp,labels=[0,1]).ravel()
    tn,fp,fn,tp=confusion_matrix(truth,pred,labels=[0,1]).ravel()
    result.append({'Model':name,'TargetValidationRecall':.70,'ThresholdFromValidation':t,'ValidationRecall':vtp/(vtp+vfn),'ValidationFPR':vfp/(vfp+vtn),'TestRecall':tp/(tp+fn),'TestFPR':fp/(fp+tn),'TestPrecision':tp/(tp+fp),'TestROC_AUC':roc_auc_score(truth,ts),'TestPR_AUC':average_precision_score(truth,ts),'TP':tp,'FP':fp,'TN':tn,'FN':fn,'TestProviders':len(truth)})
    test_scores[name]=ts
    for provider,actual,score,flag,claims in zip(df.iloc[idx_test].Provider,truth,ts,pred,df.iloc[idx_test].NumClaims):
        pred_records.append({'Provider':provider,'Model':name,'ActualProviderLabel':int(actual),'RiskScore':float(score),'ReviewFlag':int(flag),'NumClaims':int(claims),'ThresholdFromValidation':t})
metrics=pd.DataFrame(result); preds=pd.DataFrame(pred_records)
metrics.to_csv(OUT/'phase4_matched_target_model_metrics.csv',index=False)
preds.to_csv(OUT/'phase4_provider_review_simulation.csv',index=False)
print(metrics.round(4).to_string(index=False))

              Model  TargetValidationRecall  ThresholdFromValidation  ValidationRecall  ValidationFPR  TestRecall  TestFPR  TestPrecision  TestROC_AUC  TestPR_AUC  TP  FP  TN  FN  TestProviders
Logistic regression                     0.7                   0.6595             0.703         0.0815      0.8119   0.0836         0.5000       0.9593      0.7489  82  82 899  19           1082
  Gradient boosting                     0.7                   0.2248             0.703         0.0663      0.7723   0.0642         0.5532       0.9570      0.6922  78  63 918  23           1082


## 3. Paired uncertainty for model differences
Bootstrap the **same test providers** for both models; report differences without implying causal superiority. Thresholds remain fixed from validation.

In [4]:
rng=np.random.default_rng(20261118); B=600; names=list(models)
a,b=test_scores[names[0]],test_scores[names[1]]
diffs=[]
for _ in range(B):
    take=rng.integers(0,len(truth),len(truth)); yy=truth[take]
    if len(np.unique(yy))<2:continue
    diffs.append({'DeltaROC_AUC_GB_minus_LR':roc_auc_score(yy,b[take])-roc_auc_score(yy,a[take]), 'DeltaPR_AUC_GB_minus_LR':average_precision_score(yy,b[take])-average_precision_score(yy,a[take])})
diff=pd.DataFrame(diffs)
intervals=pd.DataFrame([{'Comparison':'Gradient boosting minus logistic regression','Metric':col,'Estimate':({'DeltaROC_AUC_GB_minus_LR':roc_auc_score(truth,b)-roc_auc_score(truth,a),'DeltaPR_AUC_GB_minus_LR':average_precision_score(truth,b)-average_precision_score(truth,a)})[col],'CI_Lower':diff[col].quantile(.025),'CI_Upper':diff[col].quantile(.975),'BootstrapReplicates':len(diff)} for col in diff])
intervals.to_csv(OUT/'phase4_paired_model_difference_intervals.csv',index=False)
print(intervals.round(4).to_string(index=False))

                                 Comparison                   Metric  Estimate  CI_Lower  CI_Upper  BootstrapReplicates
Gradient boosting minus logistic regression DeltaROC_AUC_GB_minus_LR   -0.0022   -0.0103    0.0050                  600
Gradient boosting minus logistic regression  DeltaPR_AUC_GB_minus_LR   -0.0567   -0.1227    0.0119                  600


## 4. Review capacity and small-provider sensitivity
Top-K is an **illustrative capacity-based ranking** and uses only test-set labels for retrospective evaluation. It does not establish investigation outcomes or monetary savings.

In [5]:
queue=[]
for name,group in preds.groupby('Model'):
    ranked=group.sort_values('RiskScore',ascending=False).reset_index(drop=True)
    for k in [25,50,100,200]:
        selected=ranked.head(k); positives=int(selected.ActualProviderLabel.sum())
        queue.append({'Model':name,'ReviewCapacity':k,'LabeledPositiveInTopK':positives,'PrecisionAtK':positives/k,'FractionOfTestPositivesCaptured':positives/int(truth.sum())})
capacity=pd.DataFrame(queue);capacity.to_csv(OUT/'phase4_review_capacity.csv',index=False)
print(capacity.round(3).to_string(index=False))
slices=[]
for name,g in preds.groupby('Model'):
    for bucket,sub in g.assign(ClaimVolumeBand=pd.cut(g.NumClaims,[-1,10,50,200,np.inf],labels=['1–10','11–50','51–200','201+'])).groupby('ClaimVolumeBand',observed=True):
        npos=int(sub.ActualProviderLabel.sum()); nneg=len(sub)-npos
        slices.append({'Model':name,'ClaimVolumeBand':str(bucket),'Providers':len(sub),'LabeledPositives':npos,'Recall':(int(((sub.ActualProviderLabel==1)&(sub.ReviewFlag==1)).sum())/npos if npos else np.nan),'FalsePositiveRate':(int(((sub.ActualProviderLabel==0)&(sub.ReviewFlag==1)).sum())/nneg if nneg else np.nan),'Interpretation':'Exploratory; unstable when few positive labels'})
slices=pd.DataFrame(slices);slices.to_csv(OUT/'phase4_claim_volume_slices.csv',index=False)
print(slices.round(3).to_string(index=False))

              Model  ReviewCapacity  LabeledPositiveInTopK  PrecisionAtK  FractionOfTestPositivesCaptured
  Gradient boosting              25                     20         0.800                            0.198
  Gradient boosting              50                     42         0.840                            0.416
  Gradient boosting             100                     63         0.630                            0.624
  Gradient boosting             200                     87         0.435                            0.861
Logistic regression              25                     24         0.960                            0.238
Logistic regression              50                     43         0.860                            0.426
Logistic regression             100                     66         0.660                            0.653
Logistic regression             200                     89         0.445                            0.881
              Model ClaimVolumeBand  Providers

## 5. Evidence boundary
Provider-level historical `PotentialFraud` labels are not adjudicated individual claims. Retrospective features may contain information unavailable before payment. This notebook does not establish Medicare/Medicaid savings, fairness, institutional adoption, deployment readiness, or generalization to future data. A prospective pilot needs an as-of feature store, independent time-split test, governance/privacy review, and payer/subject-matter evaluation.